In [1]:
import sys
import os
import json
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm

import ollama  # pip install ollama
from sentence_transformers import SentenceTransformer

# Nazwa modelu lokalnego w Ollama – uruchom wcześniej: ollama pull gemma4:12b
OLLAMA_MODEL = "gemma4:e4b"

# Model embeddingowy – zamienia tekst na wektory liczbowe (embeddingi)
# paraphrase-multilingual-mpnet-base-v2 produkuje wektory o wymiarze 768
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

print(f"Embedder gotowy. Wymiar wektora: {embedder.get_embedding_dimension()}")
print(f"Model LLM: {OLLAMA_MODEL} (obsługiwany przez Ollama)")


Embedder gotowy. Wymiar wektora: 768
Model LLM: gemma4:e4b (obsługiwany przez Ollama)


In [2]:
# Tworzymy pusty indeks FAISS typu FlatL2 – przechowuje wektory i szuka po odległości euklidesowej
# get_sentence_embedding_dimension() zwraca wymiar wektorów modelu embeddingowego (768)
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())

# Lista słowników przechowująca metadane każdego chunka (nazwa pliku, numer strony, tekst)
metadata = []

print('Number of chunks: ', index.ntotal)  # 0


Number of chunks:  0


In [3]:
class Utils:
    def __init__(self, embedding_model: SentenceTransformer = None,
                 ollama_model: str = None,
                 index=None, metadata=None, chunk_size=128):
        self.embedding_model = embedding_model
        self.ollama_model = ollama_model  # nazwa modelu w Ollama (zamiast llm_model + llm_tokenizer)
        self.index = index
        self.metadata = metadata
        # Rozmiar chunka w znakach – określa jak długie będą fragmenty tekstu
        self.chunk_size = chunk_size

    def extract_text_from_pdf(self, pdf_path):
        """
        Extract text from PDF file. Returns a list of tuples (page_number, text).
        """
        text = []
        pdf_document = pymupdf.open(pdf_path)
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            # Zamieniamy znaki nowej linii na spacje, żeby tekst był ciągły
            text.append((page_num, str(page.get_text()).replace("\n", " ")))
        return text

    def chunk_text(self, text: list[tuple[int, str]]):
        chunks = []
        for page_num, page_text in text:
            # Dzielimy tekst strony na fragmenty o długości chunk_size znaków
            page_chunks = [
                (page_num, page_text[i:i+self.chunk_size])
                for i in range(0, len(page_text), self.chunk_size)
            ]
            chunks.extend(page_chunks)
        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc="vec_db/"):
        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc="Adding chunks to FAISS")):
            # Zamieniamy tekst chunka na wektor embeddingowy
            embeddings = self.embedding_model.encode(chunk, show_progress_bar=False)
            # Dodajemy wektor do indeksu FAISS
            self.index.add(np.array([embeddings]))
            # Zapisujemy metadane chunka – powiążemy je z wektorem przez pozycję w indeksie
            self.metadata.append({
                "filename": filename,
                "page_number": page_number,
                "chunk_num": chunk_num,
                "chunk": chunk
            })
        # Zapisujemy indeks FAISS na dysk, żeby nie trzeba było go odbudowywać przy każdym uruchomieniu
        os.makedirs(db_loc, exist_ok=True)
        faiss.write_index(self.index, db_loc + "vector_database.index")
        with open(db_loc + "metadata.json", "w") as file:
            json.dump(self.metadata, file)

    def process_file(self, file_path):
        """
        Process the file and add chunks to FAISS index
        """
        if file_path.endswith('.pdf'):
            text = self.extract_text_from_pdf(file_path)
        else:
            print(f"Unsupported file format, with extension: {os.path.splitext(file_path)[1]}")
            return 0

        chunks = self.chunk_text(text)
        self.add_chunks_to_faiss(chunks, filename=os.path.basename(file_path))
        return len(chunks)

    def answer_question(self, prompt_template="", query="", max_tokens=512, temp=0.7, k=8):
        # Zamieniamy pytanie użytkownika na wektor embeddingowy
        question_embedding = self.embedding_model.encode(query, show_progress_bar=False)

        # Szukamy k najbliższych wektorów w FAISS – D to odległości, I to indeksy znalezionych chunków
        D, I = self.index.search(np.array([question_embedding]), k)
        # Pobieramy metadane (tekst) znalezionych chunków
        chunks = [self.metadata[i] for i in I[0]]

        # Sklejamy teksty chunków w jeden blok kontekstu dla modelu
        context = ""
        for i, chunk in enumerate(chunks):
            context += f"{i+1}. {chunk['chunk']}\n"

        # Wstawiamy kontekst i pytanie do szablonu promptu
        prompt = prompt_template.format(context=context, query=query)

        # Budujemy historię rozmowy – Ollama przyjmuje ten sam format messages co OpenAI
        messages = [
            {
                "role": "system",
                "content": (
                    "Be helpful, straight to the point. "
                    "Use only context. Do not hallucinate."
                )
            },
            {"role": "user", "content": prompt},
        ]

        # Wywołanie lokalnego modelu przez Ollama (zamiast transformers model.generate)
        # Ollama musi działać w tle: uruchom 'ollama serve' lub aplikację Ollama
        response = ollama.chat(
            model=self.ollama_model,
            messages=messages,
            options={
                "temperature": temp,       # im niższa tym bardziej deterministyczna odpowiedź
                "num_predict": max_tokens  # maksymalna liczba nowych tokenów do wygenerowania
            }
        )

        # Wyciągamy tekst odpowiedzi ze struktury zwróconej przez Ollama
        answer = response.message.content

        return answer, chunks


In [4]:
# Tworzymy obiekt Utils łącząc wszystkie komponenty RAG w jednym miejscu
utils = Utils(
    embedder,        # model embeddingowy do zamiany tekstu na wektory
    OLLAMA_MODEL,    # nazwa modelu Ollama (zamiast model + tokenizer)
    index,           # indeks FAISS z wektorami chunków
    metadata,        # metadane chunków (tekst, strona, plik)
    chunk_size=512   # ilość treści w każdym fragmencie - zwiększamy by dać modelowi więcej treści gdy odmówi odpowiedzi
)


In [5]:
knowledge_dir = "knowledge/"
# Przetwarzamy każdy plik PDF z katalogu – dzielimy na chunki i dodajemy do FAISS
for file in os.listdir(knowledge_dir):
    utils.process_file(knowledge_dir + file)

print('Number of chunks: ', index.ntotal)


Adding chunks to FAISS: 100%|██████████| 880/880 [00:52<00:00, 16.77it/s]

Number of chunks:  1541


In [6]:
# przykładowe pytania dotyczące powyższych dokumentów
questions = [
    # HerbAtlas (herbatlas_eng-2.pdf)
    "Jakie są właściwości lecznicze aloesu?",
    "Jak imbir jest stosowany w tradycyjnej medycynie?",
    "Jakie są wymagania uprawowe rumianku?",

    # Fizjologia roślin - Vince Ordóg (plant-physiology-vince-ordog-3.pdf)
    "Czym jest potencjał wodny roślin i jakie czynniki na niego wpływają?",
    "Jak hormony roślinne auksyny wpływają na wzrost rośliny?",
    "Co się dzieje z roślinami podczas stresu temperaturowego?",
    "Na czym polega fotosynteza u roślin C4?",

    # Wstęp do botaniki - Shipunov (introduction-to-botany-alexey-shipunov-892.pdf)
    "Jaka jest różnica między ksylemem a floemem?",
    "Czym różni się mitoza od mejozy u roślin?",
    "Jakie są główne typy tkanek roślinnych?",
]


In [9]:
from IPython.display import display, Markdown

prompt_template = """Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
Now use the following context items to answer this one user query only:
{context}
Relevant passages: 
Main User Query: {query}
Answer:\n"""

# Wybieramy losowo zapytanie z listy
random_query = np.random.choice(questions)


response, chunks = utils.answer_question(
    prompt_template=prompt_template,
    query=random_query,
    max_tokens=4096,
    temp=0.1
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Co się dzieje z roślinami podczas stresu temperaturowego?

**Odpowiedź:**

Stres temperaturowy może mieć różne formy – wysokie temperatury, niskie temperatury powyżej punktu zamarzania oraz temperatury poniżej zamarzania. Stres ten prowadzi do uszkodzenia błon i enzymów roślin.

W przypadku wysokich temperatur:
*   Procesy transportu elektronów związane z błonami stają się niestabilne, co powoduje gwałtowny spadek fotosyntezy.
*   Pod wpływem ekstremalnych warunków środowiskowych struktura białek jest wrażliwa na zaburzenia.

Rośliny mają mechanizmy obronne, takie jak indukcja białek szoku cieplnego oraz osmotyczna regulacja, które pomagają ograniczyć lub uniknąć tych problemów. Ponadto, niektóre rośliny mogą zmieniać właściwości błon lipidowych, aby utrzymać ich płynność w niższych temperaturach.

---
**Źródła:**

**[1]** `plant-physiology-vince-ordog.pdf` — strona 114

> Physiology of plant growth and  development      108    Created by XMLmind XSL-FO Converter.  heat injury to photosynthesis are more directly related to changes in membrane properties and to uncoupl...

**[2]** `plant-physiology-vince-ordog.pdf` — strona 113

> ial stomatal closure or when high relative  humidity reduces the gradient driving evaporative cooling. Increases in leaf temperature during the day can be  more pronounced in plants experiencing droug...

...

**[4]** `plant-physiology-vince-ordog.pdf` — strona 113

> ccur, depending on the magnitude and duration of the temperature  fluctuation. In this section we will discuss three types of temperature stress: high temperatures, low temperatures  above freezing, a...

**[5]** `plant-physiology-vince-ordog.pdf` — strona 7

> mones). The basic concepts of plant stress is complemented with the presentation of physiological  mechanisms against different environmental stresses....

**[6]** `plant-physiology-vince-ordog.pdf` — strona 57

> with some of the steps becoming limiting as the temperature decreases or increases.  Membrane-bound electron transport processes become unstable at high temperatures, cutting off the supply of  reduci...

**[7]** `plant-physiology-vince-ordog.pdf` — strona 117

> alline form and allows membranes to  remain fluid at lower temperatures, thus protecting the plant against damage from chilling.  A large variety of heat shock proteins can be induced by different env...

**[8]** `plant-physiology-vince-ordog.pdf` — strona 8

> uptake of water by cells generates a pressure known as turgor. Photosynthesis  requires that plants draw carbon dioxide from the atmosphere, and at the same time exposes them to water loss.  To preve...

In [11]:
random_query = "Jak bardzo radioaktywne są banany"
response, chunks = utils.answer_question(
    prompt_template=prompt_template,
    query=random_query,
    max_tokens=4096,
    temp=0.1
)


display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Jak bardzo radioaktywne są banany

**Odpowiedź:**

Na podstawie dostarczonych kontekstów nie można odpowiedzieć na pytanie, jak bardzo radioaktywne są banany.

---
**Źródła:**

**[1]** `herbatlas_eng.pdf` — strona 1

> HERBATLAS...

**[2]** `plant-physiology-vince-ordog.pdf` — strona 56

> ,000 nm), by sensible heat loss, and by evaporative (or latent) heat loss (Figure 2.18):  1. Radiative heat loss: all objects emit radiation in proportion to their temperature. However, the maximum  w...

...

**[4]** `plant-physiology-vince-ordog.pdf` — strona 54

> Under direct sunlight, PAR irradiance is about  2000 µmol m-2 s-1 (900 W m-2) at the top of a dense forest canopy, but may be only 10 µmol m-2 s-1 (4.5 W  m-2) at the bottom of the canopy. While rough...

**[5]** `herbatlas_eng.pdf` — strona 9

> ts of South America. In Costa Rica can be found in  areas with an adequate amount of moisture and shade.  They can grow to heights of 1200 meters at sea level.  Cultivation:  Reproduced by rhizomes. I...

**[6]** `plant-physiology-vince-ordog.pdf` — strona 82

> ing and a negatively charged carboxyl group.    Figure 3.11 Sructures of naturally occuring auxins (source: Taiz L., Zeiger E., 2010)  IAA is synthesized in meristems, young leaves, and developing fru...

**[7]** `plant-physiology-vince-ordog.pdf` — strona 87

> trol the level of bioactive GA and hence stem length. Such mutants have been useful in  elucidating the complex pathways of GA biosynthesis, and in determining which of the GAs in a plant has  intrins...

**[8]** `herbatlas_eng.pdf` — strona 23

> rio de salud  publica. Ecimed, 1993 •	 Projecto Plantas Medicinais. Itaipu Binacional. 2012 •	 Rodolfo Barriga Ruiz. Plantas útiles de la Amazonia  peruana: características, usos y posibilidades.  CON...